# OpenDQI Python quickstart

Three patterns in 3 minutes — same as `examples/python/{01,02,03}_*.py`, but executable cell-by-cell.

**Install:** `pip install opendqi` (v0.12.1+).

**This notebook expects** the OpenDQI repo cloned locally so the synthetic fixtures under `examples/` are reachable. For a stand-alone tutorial that downloads its own fixtures, see `docs/python.md`.

In [1]:
import json, os, shutil, subprocess
from pathlib import Path

import opendqi
print('opendqi version:', opendqi.__version__)

# Repo root (notebook is at examples/python/quickstart.ipynb)
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'Cargo.toml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
print('repo root:', REPO_ROOT)

opendqi version: 0.15.1
repo root: /Users/paul/Desktop/opendqi


## Pattern 1 — Scan a normalized Parquet

The simplest entry point: hand `scan_parquet` a path to a canonical EMIR Parquet (produced by `opendqi emir normalize`). Get back a `summary` dict and an `issues` pyarrow.Table.

Here we generate the Parquet on the fly via the CLI binary.

In [2]:
csv = REPO_ROOT / 'examples' / 'emir' / 'sample.csv'
mapping = REPO_ROOT / 'examples' / 'emir' / 'sample_mapping.yml'
parquet = REPO_ROOT / 'target' / 'examples' / 'sample.parquet'
parquet.parent.mkdir(parents=True, exist_ok=True)

if not parquet.exists():
    bin_path = shutil.which('opendqi') or str(REPO_ROOT / 'target' / 'debug' / 'opendqi')
    subprocess.run(
        [bin_path, 'emir', 'normalize', str(csv),
         '--mapping', str(mapping), '--out', str(parquet)],
        check=True, env={**os.environ, 'RUST_LOG': 'warn'},
    )

result = opendqi.emir.scan_parquet(str(parquet))
print(json.dumps(result.summary, indent=2, default=str))

{
  "regime": "emir",
  "files_processed": 1,
  "records_processed": 8,
  "issues_total": 97,
  "issues_by_severity": {
    "warning": 58,
    "high": 37,
    "critical": 2
  },
  "issues_by_dimension": {
    "completeness": 50,
    "uniqueness": 2,
    "timeliness": 8,
    "validity": 25,
    "accuracy": 10,
    "consistency": 2
  },
  "quality_score": 25.75,
  "started_at": "2026-05-21T21:20:59.873012+00:00",
  "finished_at": "2026-05-21T21:20:59.880377+00:00"
}


## Pattern 2 — XML → `scan_table` (no Parquet roundtrip)

When you already have ISO 20022 XML in memory, `parse_xml` produces the canonical Arrow Table directly; `scan_table` then runs the same check suite as `scan_parquet`.

In [3]:
xml = REPO_ROOT / 'examples' / 'quickstart-emir' / 'auth030-tar.xml'

table = opendqi.emir.parse_xml(str(xml))
print(f'parsed {table.num_rows} record(s), {len(table.column_names)} columns')

mapping = {n: n for n in table.column_names}   # identity
result = opendqi.emir.scan_table(table, mapping)
print(json.dumps(result.summary, indent=2, default=str))

parsed 20 record(s), 55 columns
{
  "regime": "emir",
  "files_processed": 1,
  "records_processed": 20,
  "issues_total": 197,
  "issues_by_severity": {
    "warning": 86,
    "high": 106,
    "critical": 5
  },
  "issues_by_dimension": {
    "completeness": 180,
    "uniqueness": 5,
    "consistency": 12
  },
  "quality_score": 27.849998474121094,
  "started_at": "2026-05-21T21:21:00.074342+00:00",
  "finished_at": "2026-05-21T21:21:00.074686+00:00"
}


## Pattern 3 — Custom column mapping (the warehouse path)

When your Arrow table comes from a custom source, the column names won't match the canonical EMIR field names. The `mapping` dict reroutes each canonical field to your actual column name.

Below: a small Arrow groupby on `result.issues` showing the top 5 check IDs — pure Arrow, no pandas needed.

In [4]:
rename = {
    'uti':                 'TradeUTI',
    'valuation_timestamp': 'MtmTs',
    'maturity_date':       'ContractEnd',
}
user_table = table.rename_columns([rename.get(n, n) for n in table.column_names])

mapping = {
    **{name: name for name in user_table.column_names if name not in rename.values()},
    'uti':                 'TradeUTI',
    'valuation_timestamp': 'MtmTs',
    'maturity_date':       'ContractEnd',
}
result = opendqi.emir.scan_table(user_table, mapping)
print(f'records={result.summary["records_processed"]} '
      f'issues={result.summary["issues_total"]} '
      f'score={result.summary["quality_score"]:.2f}')

from collections import Counter
top_checks = Counter(result.issues.column('check_id').to_pylist()).most_common(5)
print('\nTop 5 check IDs:')
for check_id, n in top_checks:
    print(f'  {n:>3}x {check_id}')

records=20 issues=197 score=27.85



Top 5 check IDs:
   20x EMIR.COMP.ASSET_CLASS_MISSING
   20x EMIR.COMP.CLEARING_STATUS_MISSING
   20x EMIR.COMP.COUNTERPARTY_1_MISSING
   20x EMIR.COMP.COUNTERPARTY_2_MISSING
   20x EMIR.COMP.INTRAGROUP_INDICATOR_MISSING


## Pattern 7 — EMIR Data Quality Pack (24 indicators)

The headline DQI feature. Above the 216 granular checks sits an **aggregated layer** with 24 regulator-style EMIR indicators (numerator / denominator / rate / threshold / status) + drill-down evidence. Same scan, two views — committee-readable on top, forensic underneath. v0.16 grew this from 10 → 24 EMIR indicators (+ 4 SFTR — see Pattern 8 below).

Run on the 5-layer kit at `examples/emir-data-quality-pack/`. The result has 4 fields:
- `result.indicators` — pyarrow.Table, 24 rows × 11 cols (v1.0 stable)
- `result.evidence`   — pyarrow.Table, ≤ 480 rows × 7 cols (v1.0 stable)
- `result.issues`     — pyarrow.Table, granular (same contract as v0.12+)
- `result.summary`    — dict, same shape as `summary.json`

**Disclaimer**: DQI verdicts are **internal data quality indicators**, not regulatory verdicts. A `red` status is an internal alert, not a declaration of non-compliance — see [`docs/data-quality-pack.md`](../../docs/data-quality-pack.md).

In [5]:
kit = REPO_ROOT / 'examples' / 'emir-data-quality-pack'

pack = opendqi.emir.data_quality_pack(
    tsr=str(kit / 'tsr.xml'),
    tar=str(kit / 'tar.xml'),
    msr=str(kit / 'msr.xml'),
    mar=str(kit / 'mar.xml'),
    feedback=str(kit / 'feedback.xml'),
    as_of='2026-05-21',
)
print(repr(pack))

# 24 EMIR indicators — committee-readable
ind_df = pack.indicators.select(
    ['indicator_id', 'status', 'numerator', 'denominator', 'rate']
).to_pandas()
ind_df['rate'] = ind_df['rate'].apply(
    lambda r: f'{r * 100:.2f}%' if r is not None else ''
)
print('\n=== indicators ===')
print(ind_df.to_string(index=False))

# Drill-down evidence — top offenders
print(f'\n=== evidence ({pack.evidence.num_rows} rows) ===')
print(pack.evidence.select(['indicator_id', 'uti', 'explanation']).to_pandas().head(8).to_string(index=False))

PyDqiPackResult(indicators=13/24 computed, evidence=76, issues=224, score=54.32)



=== indicators ===
                        indicator_id         status  numerator  denominator    rate
                    DQI_ANOMALY_RATE          amber          1            8  12.50%
                    DQI_COL_ALL_ZERO            red          1            4  25.00%
               DQI_COL_MISSING_STATE            red          1            1 100.00%
                 DQI_COL_STALE_STATE            red          1            4  25.00%
                    DQI_CONF_MISSING not_applicable          0            0    nan%
               DQI_DUPLICATE_REPORTS            red          2            8  25.00%
                     DQI_ERR_MISSING            red         20           20 100.00%
             DQI_FIELD_MISMATCH_RATE not_applicable          0            0    nan%
                     DQI_LEI_MISSING          green          0            8   0.00%
DQI_MARGIN_INCONSISTENT_POST_HAIRCUT not_applicable          0            0    nan%
 DQI_MARGIN_INCONSISTENT_PRE_HAIRCUT not_applicable     

## Pattern 8 — SFTR Data Quality Pack (v0.16+, 4 indicators)

Sister API on the SFTR side. v0.16 ships **4 SFTR indicators** on the T2 layer of `auth.079` (TSR) + `auth.052` (TAR) :
- `DQI_COLLATERAL_VALUE_MISSING_SFTR` (completeness)
- `DQI_LOAN_VALUE_MISSING_SFTR` (completeness)
- `DQI_LOAN_VALUE_STALE_SFTR` (timeliness — TARGET2 business days)
- `DQI_TIM_REPORTING_LATE_SFTR` (timeliness)

**Paths-only in v0.16** (no pyarrow.Table dual-input on the SFTR side yet — v0.17). T3 margin / `auth.080` reconciliation / `auth.083` missing-collateral indicators are scheduled for v0.17 too. Same `PyDqiPackResult` shape as Pattern 7 — same 4 fields, same v1.0 Arrow schemas.

In [6]:
sftr_kit = REPO_ROOT / 'examples' / 'sftr-data-quality-pack'

sftr_pack = opendqi.sftr.data_quality_pack(
    tsr=str(sftr_kit / 'tsr.xml'),
    tar=str(sftr_kit / 'tar.xml'),
    as_of='2026-05-21',
)
print(repr(sftr_pack))

# 4 SFTR indicators
sftr_df = sftr_pack.indicators.select(
    ['indicator_id', 'status', 'numerator', 'denominator', 'rate']
).to_pandas()
sftr_df['rate'] = sftr_df['rate'].apply(
    lambda r: f'{r * 100:.2f}%' if r is not None else ''
)
print('\n=== SFTR indicators ===')
print(sftr_df.to_string(index=False))

PyDqiPackResult(indicators=4/4 computed, evidence=14, issues=82, score=77.06)

=== SFTR indicators ===
                     indicator_id status  numerator  denominator    rate
DQI_COLLATERAL_VALUE_MISSING_SFTR  green          0           10   0.00%
      DQI_LOAN_VALUE_MISSING_SFTR  amber          1           13   7.69%
        DQI_LOAN_VALUE_STALE_SFTR    red         13           13 100.00%
      DQI_TIM_REPORTING_LATE_SFTR  green          0           14   0.00%


## Where to go next

- [`docs/data-quality-pack.md`](../../docs/data-quality-pack.md) — full DQI spec (24 EMIR + 4 SFTR indicators, thresholds, Arrow schemas, disclaimer)
- [`docs/python.md`](../../docs/python.md) — full quickstart with DuckDB / Polars / pandas / Spark integration patterns
- [`docs/python-roadmap.md`](../../docs/python-roadmap.md) — architecture spec + v0.13+ roadmap
- [`README.md`](../../README.md) — project overview, CLI + UI surface, 216 checks coverage + the v0.15 Data Quality Pack

**Status: preview (v0.16.x).** The v1.0 Arrow contracts for `result.issues` (11 cols, locked v0.12.0), `result.indicators` (11 cols, locked v0.15.0), and `result.evidence` (7 cols, locked v0.15.0) are **stable**; v0.16 expanded the DQI surface from 10 → 28 indicators by adding *rows* only, never new columns. The API may grow additively in v0.17 (SFTR T3 / reconciliation / missing-collateral indicators, CLI/Python wiring of auth.091 cross-CP DQIs, dual-input pyarrow.Table on the SFTR side, threshold profile presets).